#### Load & Split Data

In [1]:
import re
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [2]:
docx_loader = Docx2txtLoader('../../data/Introduction_to_Data_and_Data_Science.docx')
docs = docx_loader.load()

for doc in docs:
    doc.page_content = re.sub('\n\n', '<<PARA>>', doc.page_content)

    doc.page_content = re.sub('\n', ' ', doc.page_content)

    doc.page_content = re.sub('<<PARA>>', '\n\n', doc.page_content)

In [3]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [('#', 'Document Title'), ('##', 'Topic Title')])

for doc in docs:
    md_chunks = md_splitter.split_text(doc.page_content)

In [4]:
rec_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50, separators = ['\n\n', '\n', ' ', ''])

chunks = rec_splitter.split_documents(md_chunks)

#### Embedding Function

In [5]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [6]:
embedding_fn = HuggingFaceEmbeddings(model_name = 'BAAI/bge-base-en-v1.5')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### Embed & Store Data

> embed each chunk into an embedding vector`

> can't use relation db as they only look for data matching the input exactly in the db

> vector db are optimized to look for data that is similar to the input

In [7]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [8]:
vectorstore = Chroma.from_documents(documents = chunks, embedding = embedding_fn, persist_directory = '../../vectordb/intro_to_data_science') # re-running this line will not overwrite the existing db but duplicate the existing entries

# vectorstore_from_directory = Chroma(embedding_function = embedding_fn, persist_directory = '../../vectordb/intro_to_data_science')

> embedding fn is necessary when loading from directory as we need the same fn for embedding the user query as well as updating the existing vector db

##### Add New Document

In [9]:
new_doc = Document(metadata = {'Document Title': 'New Document', 'Topic Title': 'New Topic'}, page_content = "New Text")

vectorstore.add_documents([new_doc])

['fed31aca-e709-4d47-b65e-b1913ba12a79']

In [10]:
vectorstore.get('fed31aca-e709-4d47-b65e-b1913ba12a79') # by default, only the keys 'metadatas' and 'documents' have values displayed

{'ids': ['fed31aca-e709-4d47-b65e-b1913ba12a79'],
 'embeddings': None,
 'documents': ['New Text'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Document Title': 'New Document', 'Topic Title': 'New Topic'}]}

##### Update Document

In [11]:
updated_doc = Document(metadata = {'Document Title': 'Updated Document', 'Topic Title': 'Updated Topic'}, page_content = "Updated Text")

vectorstore.update_document(document_id = 'fed31aca-e709-4d47-b65e-b1913ba12a79', document = updated_doc)

In [12]:
vectorstore.get(ids = 'fed31aca-e709-4d47-b65e-b1913ba12a79')

{'ids': ['fed31aca-e709-4d47-b65e-b1913ba12a79'],
 'embeddings': None,
 'documents': ['Updated Text'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Topic Title': 'Updated Topic',
   'Document Title': 'Updated Document'}]}

##### Delete Document

In [13]:
vectorstore.delete(ids = ['fed31aca-e709-4d47-b65e-b1913ba12a79'])

In [14]:
vectorstore.get(ids = ['fed31aca-e709-4d47-b65e-b1913ba12a79'])

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}